# LIBRARIES DEFINITION

In [ ]:
import os
import math
import scipy.io
import pathlib  #Útil para trabajar con directorios

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib as plt

from joblib import dump, load  #Útil para salvar classificadores
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

#Nuevo
from numpy.lib.stride_tricks import sliding_window_view
import gc
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

import time

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier 
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
import keras
keras.config.enable_unsafe_deserialization()
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import random

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism() 

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# FUNCTIONS DEFINITIONS

## RL STAGE

In [ ]:
def createFolder(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

In [ ]:
def trainRepresentationPCA(X=[], d=2, name='PCA', save_filename=''):
  pca   = PCA(n_components=d,svd_solver='covariance_eigh',random_state=42) #Considerar el parámetro para trabajar con covarianzas

  start = time.process_time_ns()
  pca.fit(X)
  end   = time.process_time_ns()

  dump(pca, save_filename)

  elapsed_time = (end - start)*1e-9

  return pca, elapsed_time

In [ ]:
def extract_spatial_patches(I, patch_size=3):
    """
    Transforma una imagen (M, N, B) en una matriz de parches (Np, patch_size*patch_size*B).
    """
    M, N, B = I.shape
    offset = patch_size // 2

    # Se añade un marco de ceros proporcional al tamaño del parche
    I_padded = np.pad(I, ((offset, offset), (offset, offset), (0, 0)), mode='constant')

    # Info_del_parche = ancho * alto * bandas
    patch_dim = patch_size * patch_size * B
    J_patches = np.zeros((M * N, patch_dim))

    # Usamos los índices originales para recorrer la imagen acolchada
    count = 0
    for r in range(M):
        for c in range(N):
            # Extraemos el bloque centrado en (r, c) de la imagen con padding
            patch = I_padded[r : r + patch_size, c : c + patch_size, :]
            # Aplanamos el parche (bloque -> vector) y lo guardamos en la fila
            J_patches[count, :] = patch.flatten()
            count += 1

    return J_patches

In [ ]:
def trainRepresentationAE_original(X_T=[], X_v=[], input_dim=100, encoder_dims=2, lambda_=0.01, name='HOAE', save_filename=''):
  # Define the HOAE architecture
  inputs  = tf.keras.Input(shape=(input_dim,), name="Inputs")
  encoder = tf.keras.layers.Dense(units=encoder_dims,
                                  activation='relu',
                                  use_bias=True,
                                  kernel_initializer=tf.keras.initializers.HeNormal(),
                                  bias_initializer='zeros',
                                  kernel_regularizer=tf.keras.regularizers.OrthogonalRegularizer(factor=lambda_, mode="rows"),
                                  name="Encoder")(inputs)

  decoder = tf.keras.layers.Dense(units=input_dim,
                                  activation='sigmoid',
                                  use_bias=True,
                                  kernel_initializer=tf.keras.initializers.GlorotNormal(),
                                  bias_initializer='zeros',
                                  name="Decoder")(encoder)

  HOAE = tf.keras.Model(inputs=inputs, outputs=decoder, name=name)
  HOAE.compile(optimizer=tf.keras.optimizers.Adam(),
               loss=tf.keras.losses.MeanSquaredError(),
               metrics=[tf.keras.losses.MeanSquaredError()])

  # Define HOAE callbacks
  ckpt_callback = tf.keras.callbacks.ModelCheckpoint(filepath=save_filename,
                                                      monitor="val_loss",
                                                      mode="min",
                                                      save_best_only=True)

  early_callback= tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", verbose=0, patience=25, restore_best_weights=True)

  debug_callback= tf.keras.callbacks.LambdaCallback(
      on_epoch_end=lambda epoch, logs: print(f"Epoch {epoch+1:03d}/1000 - loss: {logs['loss']:.6f} - val_loss: {logs['val_loss']:.6f}") if (epoch+1) % 1000 == 0 else None
  )


  train_dataset = tf.data.Dataset.from_tensor_slices((X_T, X_T))
  train_dataset = train_dataset.shuffle(buffer_size=len(X_T),seed=42).batch(128).prefetch(tf.data.AUTOTUNE)

  val_dataset = tf.data.Dataset.from_tensor_slices((X_v, X_v))
  val_dataset = val_dataset.batch(128).prefetch(tf.data.AUTOTUNE)

  start = time.perf_counter()

  # Entrenamiento usando los datasets optimizados
  history = HOAE.fit(train_dataset,
                     epochs=1000,
                     verbose=0,
                     validation_data=val_dataset,
                     callbacks=[early_callback, ckpt_callback, debug_callback])
  
  end = time.perf_counter()
  elapsed_time = end - start

  HOAE.evaluate(val_dataset, verbose=0)
  HOAE.save(save_filename)

  return HOAE, elapsed_time, history

In [ ]:
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, num_tokens, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_tokens = num_tokens
        self.embed_dim = embed_dim

    def build(self, input_shape):
        # El número de tokens aumenta en 1 debido al [CLS] token
        self.pos_emb = self.add_weight(
            name="pos_emb",
            shape=(1, self.num_tokens + 1, self.embed_dim),
            initializer=tf.keras.initializers.RandomNormal(stddev=0.02),
            trainable=True
        )

    def call(self, inputs):
        return inputs + self.pos_emb

class CLSToken(tf.keras.layers.Layer):
    """Añade un token [CLS] aprendible al inicio de la secuencia."""
    def __init__(self, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim

    def build(self, input_shape):
        self.cls_token = self.add_weight(
            name="cls_token",
            shape=(1, 1, self.embed_dim),
            initializer=tf.keras.initializers.RandomNormal(stddev=0.02),
            trainable=True
        )

    def call(self, inputs):
        # Concatenar el token [CLS] al inicio de la dimensión de la secuencia
        batch_size = tf.shape(inputs)[0]
        broadcast_cls = tf.broadcast_to(self.cls_token, [batch_size, 1, self.embed_dim])
        return tf.concat([broadcast_cls, inputs], axis=1)

In [ ]:
def trainViT(X_T, X_v, num_tokens=9, input_dim=103, 
             embed_dim=48, num_heads=2, ff_dim=96, dropout_rate=0.1,
             latent_dim=32, name='ViT', save_filename='vit_original_ae.keras'):
    """
    Arquitectura ViT adaptada para Representación de Imágenes Hiperespectrales.
    """
    
    # --- ARQUITECTURA (ENCODER + DECODER) ---
    inputs = tf.keras.Input(shape=(num_tokens, input_dim), name="Patch_Input")
    
    # Encoder
    x = tf.keras.layers.Dense(units=embed_dim, name="Linear_Projection")(inputs)
    x = CLSToken(embed_dim=embed_dim, name="Add_CLS_Token")(x)
    x = PositionalEmbedding(num_tokens=num_tokens, embed_dim=embed_dim, name="Pos_Emb")(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    norm_1 = tf.keras.layers.LayerNormalization(epsilon=1e-6, name="Norm_1")(x)
    attn_output = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim, dropout=dropout_rate, name="MSA"
    )(norm_1, norm_1)
    x = tf.keras.layers.Add(name="Residual_1")([x, attn_output])

    norm_2 = tf.keras.layers.LayerNormalization(epsilon=1e-6, name="Norm_2")(x)
    ffn_output = tf.keras.layers.Dense(units=ff_dim, activation='gelu', name="FFN_GELU")(norm_2)
    ffn_output = tf.keras.layers.Dropout(dropout_rate)(ffn_output)
    ffn_output = tf.keras.layers.Dense(units=embed_dim, name="FFN_Proj")(ffn_output)
    x = tf.keras.layers.Add(name="Residual_2")([x, ffn_output])

    # Latent Space (Z)
    cls_output = tf.keras.layers.Lambda(lambda t: t[:, 0], name="Extract_CLS")(x)
    norm_cls = tf.keras.layers.LayerNormalization(epsilon=1e-6, name="Norm_CLS")(cls_output)
    latent_z = tf.keras.layers.Dense(units=latent_dim, name="Latent_Space_Z")(norm_cls)

    # Decoder
    dec = tf.keras.layers.Dense(units=embed_dim, activation='gelu', name="Decoder_Init")(latent_z)
    dec = tf.keras.layers.RepeatVector(num_tokens, name="Repeat_Latent")(dec)
    outputs = tf.keras.layers.Dense(units=input_dim, activation='sigmoid', name="Reconstruction")(dec)

    # --- COMPILACIÓN ---
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name=name)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss=tf.keras.losses.MeanSquaredError())

    # --- CALLBACKS ---
    ckpt_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=save_filename,
        monitor="val_loss",
        mode="min",
        save_best_only=True
    )

    early_callback = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", 
        mode="min", 
        verbose=0, 
        patience=30, 
        restore_best_weights=True
    )

    debug_callback = tf.keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs: print(
            f"Epoch {epoch+1:03d}/1000 - loss: {logs['loss']:.6f} - val_loss: {logs['val_loss']:.6f}"
        ) if (epoch+1) % 1000 == 0 else None
    )

    # --- PIPELINE DE DATOS ---
    train_dataset = tf.data.Dataset.from_tensor_slices((X_T, X_T))
    train_dataset = train_dataset.shuffle(buffer_size=len(X_T),seed=42).batch(128).prefetch(tf.data.AUTOTUNE)
    val_dataset = tf.data.Dataset.from_tensor_slices((X_v, X_v)).batch(128).prefetch(tf.data.AUTOTUNE)

    # --- ENTRENAMIENTO Y GUARDADO ---
    start = time.perf_counter()
    history = model.fit(
        train_dataset, 
        epochs=1000, 
        verbose=0, 
        validation_data=val_dataset, 
        callbacks=[early_callback, ckpt_callback, debug_callback]
    )
    end = time.perf_counter()
    elapsed_time = end - start

    # Evaluación y guardado final explícito
    model.evaluate(val_dataset, verbose=0)
    model.save(save_filename)

    return model, elapsed_time, history

In [ ]:
class PatchMasking(tf.keras.layers.Layer):
    """Elimina físicamente una proporción de tokens (píxeles)."""
    def __init__(self, mask_ratio=0.75, **kwargs):
        super().__init__(**kwargs)
        self.mask_ratio = mask_ratio

    def call(self, inputs):
        # inputs: (batch, 9, embed_dim)
        batch_size = tf.shape(inputs)[0]
        num_tokens = tf.shape(inputs)[1]
        num_masked = tf.cast(tf.math.round(tf.cast(num_tokens, tf.float32) * self.mask_ratio), tf.int32)
        
        # Generar índices aleatorios y seleccionar solo los visibles
        rand_indices = tf.argsort(tf.random.uniform(shape=(batch_size, num_tokens)), axis=-1)
        visible_indices = rand_indices[:, num_masked:]
        
        # Recolectar tokens visibles (Gather)
        visible_tokens = tf.gather(inputs, visible_indices, batch_dims=1)
        return visible_tokens, visible_indices

In [ ]:
def trainMAE_ViT(X_T, X_v, num_tokens=9, input_dim=103, 
                 embed_dim=64, latent_dim=64, mask_ratio=0.5,
                 name='MAE_ViT_HSI', save_filename='mae_vit.keras'):
    
    # --- PIPELINE DE DATOS ---
    batch_size = 128
    train_dataset = tf.data.Dataset.from_tensor_slices((X_T, X_T))
    train_dataset = train_dataset.shuffle(buffer_size=len(X_T),seed=42).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_v, X_v))
    val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # --- ARQUITECTURA MAE ---
    inputs = tf.keras.Input(shape=(num_tokens, input_dim), name="Patch_Input")
    
    # Encoder (tokens visibles solamente)
    x = tf.keras.layers.Dense(embed_dim)(inputs) 
    visible_tokens, visible_indices = PatchMasking(mask_ratio=mask_ratio)(x)
    
    norm_1 = tf.keras.layers.LayerNormalization()(visible_tokens)
    attn = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=embed_dim)(norm_1, norm_1)
    x = tf.keras.layers.Add()([visible_tokens, attn])
    
    latent_z = tf.keras.layers.GlobalAveragePooling1D(name="Latent_Space_Z")(x)
    latent_z = tf.keras.layers.Dense(latent_dim)(latent_z)

    # Decoder (reconstrucción del parche completo) 
    dec = tf.keras.layers.RepeatVector(num_tokens)(latent_z)
    dec = tf.keras.layers.Dense(embed_dim, activation='gelu')(dec)
    outputs = tf.keras.layers.Dense(input_dim, activation='sigmoid', name="Reconstruction")(dec)

    MAE_Model = tf.keras.Model(inputs=inputs, outputs=outputs, name=name)
    MAE_Model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')

    # --- CALLBACKS (Idénticos a la metodología original) ---
    ckpt_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=save_filename,
        monitor="val_loss",
        mode="min",
        save_best_only=True
    )

    early_callback = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", 
        mode="min", 
        verbose=0, 
        patience=30, 
        restore_best_weights=True
    )

    debug_callback = tf.keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs: print(
            f"Epoch {epoch+1:03d}/1000 - loss: {logs['loss']:.6f} - val_loss: {logs['val_loss']:.6f}"
        ) if (epoch+1) % 1000 == 0 else None
    )

    # --- ENTRENAMIENTO ---
    start = time.perf_counter()
    history = MAE_Model.fit(
        train_dataset, 
        epochs=1000, 
        validation_data=val_dataset, 
        verbose=0, 
        callbacks=[early_callback, ckpt_callback, debug_callback]
    )
    end = time.perf_counter()
    elapsed_time = end - start

    # Guardado explícito 
    MAE_Model.evaluate(val_dataset, verbose=0)
    MAE_Model.save(save_filename)

    return MAE_Model, elapsed_time, history

In [ ]:
def extract_spatial_patches_vectorized(I, patch_size=3):
    """
    Versión vectorizada y optimizada para la extracción de parches secuenciales.
    Evita los bucles 'for' anidados, acelerando drásticamente el procesamiento.
    """
    M, N, B = I.shape
    offset = patch_size // 2

    # Aplicación del padding
    I_padded = np.pad(I, ((offset, offset), (offset, offset), (0, 0)), mode='constant')

    # sliding_window_view extrae todos los parches simultáneamente.
    # La ventana de extracción es (patch_size, patch_size, B).
    # La forma resultante del tensor 'patches' es: (M, N, 1, patch_size, patch_size, B)
    patches = sliding_window_view(I_padded, window_shape=(patch_size, patch_size, B))

    # Remodelamos directamente a la forma final requerida por el Transformer:
    # (Total de píxeles, Número de tokens, Bandas) -> (M*N, patch_size*patch_size, B)
    J_patches = patches.reshape(M * N, patch_size * patch_size, B)

    return J_patches

In [ ]:
def trainDINO_ViT(X_T, X_v, num_tokens=9, input_dim=103, 
                  encoder_dims=64, embed_dim=128, 
                  name='DINO_ViT', save_filename='dino_model.keras'):
    """
    Versión DINO con paridad total con la función AE original. 
    Incluye Early Stopping, persistencia del Teacher y métricas consistentes. [cite: 51, 130]
    """
    
    # --- TRANSFORMACIONES (Espacial, Espectral y Oclusión) ---
    @tf.function
    def augment(images, apply_occlusion=False):
        # Invarianza Espacial 
        x = tf.reshape(images, [-1, 3, 3, input_dim])
        x = tf.image.random_flip_left_right(x)
        x = tf.image.rot90(x, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
        
        # Invarianza Espectral (Ruido y Escala) 
        scale = tf.random.uniform([tf.shape(x)[0], 1, 1, 1], 0.99, 1.01)
        x = (x * scale) + tf.random.normal(tf.shape(x), stddev=0.01)
        x = tf.reshape(x, [-1, 9, input_dim])
        
        if apply_occlusion:
            mask = np.ones((1, 9, 1), dtype=np.float32)
            mask[:, 4, :] = 0.0 # Ocluir píxel central para aprendizaje contextual
            x = x * mask
        return x

    # --- ARQUITECTURA (ViT Backbone) ---
    def build_vit():
        inputs = tf.keras.Input(shape=(num_tokens, input_dim))
        x = tf.keras.layers.Dense(embed_dim)(inputs)
        x = CLSToken(embed_dim)(x) 
        x = PositionalEmbedding(num_tokens, embed_dim)(x) 
        
        norm = tf.keras.layers.LayerNormalization()(x)
        attn = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=embed_dim)(norm, norm)
        x = tf.keras.layers.Add()([x, attn])
        
        cls_extract = tf.keras.layers.Lambda(lambda t: t[:, 0], name="CLS_Extract")(x)
        latent = tf.keras.layers.Dense(encoder_dims, name="Latent_Space")(cls_extract)
        return tf.keras.Model(inputs, latent)

    student = build_vit()
    teacher = build_vit()
    teacher.set_weights(student.get_weights())

    K_DIM = 4096
    def build_head():
        return tf.keras.Sequential([
            tf.keras.layers.Dense(512, activation='gelu'),
            tf.keras.layers.Dense(K_DIM)
        ])
    
    s_head, t_head = build_head(), build_head()
    t_head.set_weights(s_head.get_weights())

    # --- CONFIGURACIÓN DE ENTRENAMIENTO ---
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    teacher_center = tf.Variable(tf.zeros((1, K_DIM)), trainable=False)
    
    @tf.function
    def train_step(batch):
        v_teacher = augment(batch, apply_occlusion=False)
        v_student = augment(batch, apply_occlusion=True)
        
        with tf.GradientTape() as tape:

            t_logits = t_head(teacher(v_teacher, training=False))
            s_logits = s_head(student(v_student, training=True))
            t_probs = tf.nn.softmax((t_logits - teacher_center) / 0.01)
            s_probs = tf.nn.log_softmax(s_logits / 0.1)
            loss = tf.reduce_mean(-tf.reduce_sum(t_probs * s_probs, axis=-1))

        grads = tape.gradient(loss, student.trainable_variables + s_head.trainable_variables)
        optimizer.apply_gradients(zip(grads, student.trainable_variables + s_head.trainable_variables))

        # Update Maestro via EMA (Exponential Moving Average) 
        for s_v, t_v in zip(student.variables + s_head.variables, teacher.variables + t_head.variables):
            t_v.assign(t_v * 0.999 + s_v * 0.001)
            
        teacher_center.assign(teacher_center * 0.9 + tf.reduce_mean(t_logits, axis=0, keepdims=True) * 0.1)
        return loss

    batch_size = 512
    # --- PIPELINE DE DATOS --- 
    train_ds = tf.data.Dataset.from_tensor_slices(X_T.astype('float32')).shuffle(buffer_size=len(X_T),seed=42).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    val_ds = tf.data.Dataset.from_tensor_slices(X_v.astype('float32')).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    # --- CICLO DE ENTRENAMIENTO ---
    start = time.perf_counter()
    best_val_loss = float('inf')
    patience = 50
    wait = 0
    history_dict = {'loss': [], 'val_loss': []}

    for epoch in range(1000):
        train_losses = []
        for batch in train_ds:
            loss = train_step(batch)
            train_losses.append(loss)
        
        # Validación interna para Early Stopping 
        val_losses = []
        for v_batch in val_ds:
            v_t_logits = t_head(teacher(v_batch, training=False))
            v_s_logits = s_head(student(v_batch, training=False))
            v_loss = tf.reduce_mean(-tf.reduce_sum(tf.nn.softmax(v_t_logits/0.02) * tf.nn.log_softmax(v_s_logits/0.1), axis=-1))
            val_losses.append(v_loss)

        cur_train_loss = np.mean(train_losses)
        cur_val_loss = np.mean(val_losses)
        
        history_dict['loss'].append(cur_train_loss)
        history_dict['val_loss'].append(cur_val_loss)

        # Debug report cada 100 épocas 
        if (epoch + 1) % 500 == 0:
            print(f"Epoch {epoch+1:03d}/1000 - loss: {cur_train_loss:.6f} - val_loss: {cur_val_loss:.6f}")

        # Lógica de Checkpoint y Early Stopping [
        if cur_val_loss < best_val_loss:
            best_val_loss = cur_val_loss
            wait = 0
            teacher.save(save_filename) # Guardar el mejor Teacher Backbone
        else:
            wait += 1
            if wait >= patience:
                #print(f"Early stopping en época {epoch+1}")
                break

    elapsed_time = time.perf_counter() - start
    
    # Evaluación final y guardado explícito 
    teacher.save(save_filename)
    
    class History: pass
    history = History(); history.history = history_dict

    return teacher, elapsed_time, history

## CLASIFICATION STAGE

In [ ]:
@tf.keras.utils.register_keras_serializable(package="MyLayers")
class TiedDense(tf.keras.layers.Layer):
  def __init__(self, tied_to, units=2, activation='relu', use_bias=True, *args, **kwargs):
    super(TiedDense, self).__init__(*args, **kwargs)
    self.tied_to    = tied_to
    self.units      = units
    self.activation = tf.keras.activations.get(activation)
    self.use_bias   = use_bias

  def build(self, input_shape):
    self.kernel = self.add_weight(shape=(input_shape[-1], self.units),
                                  initializer=None,
                                  trainable=False,
                                  name="kernel")

    self.bias   = self.add_weight(shape=(self.units,),
                                  initializer="zeros",
                                  trainable=self.use_bias,
                                  name="bias")

  def call(self, inputs):
    self.kernel.assign(tf.linalg.matrix_transpose(self.tied_to.kernel))

    return self.activation(tf.linalg.matmul(inputs, self.kernel) + self.bias)

  def get_config(self):
    base_config = super(TiedDense, self).get_config()
    config      = {"units"     : self.units,
                   "activation": self.activation,
                   "use_bias"  : self.use_bias,
                   "tied_to"   : tf.keras.utils.serialize_keras_object(self.tied_to)}

    return {**base_config, **config}

  @classmethod
  def from_config(cls, config):
    tied_to_config = config.pop("tied_to")
    tied_to = tf.keras.utils.deserialize_keras_object(tied_to_config)
    return cls(tied_to, **config)

In [ ]:
def loadRepresentationModel(_root_model="", _model_type="PCA", _k=0.25, _d=2, _lambda=0.00):
  model   = []

  match _model_type:
    case "PCA":

      filename= _root_model + "_Representation_PCA" + "_k_{:.2f}".format(_k).replace('.','') + "_d_{:02d}".format(_d) + ".joblib"
      model   = load(filename)
    
    case "AE":
      _lambda = 0.00 #Coefficient factor for HOAE. If 0.00 it behaves as a normal autoencoder
      filename= _root_model + "_Representation_HOAE" + "_k_{:.2f}".format(_k).replace('.','') + "_d_{:02d}".format(_d) + "_f_{:.3f}".format(_lambda).replace('.','') + ".keras"
      modelkrs= tf.keras.models.load_model(filename)
      inputs  = modelkrs.inputs
      outputs = modelkrs.layers[-2].output
      model   = tf.keras.Model(inputs=inputs, outputs=outputs)

    case "AE_Patch":
      _lambda = 0.00 #Coefficient factor for HOAE. If 0.00 it behaves as a normal autoencoder
      filename= _root_model + "_Representation_HOAE_Patch" + "_k_{:.2f}".format(_k).replace('.','') + "_d_{:02d}".format(_d) + "_f_{:.3f}".format(_lambda).replace('.','') + ".keras"
      modelkrs= tf.keras.models.load_model(filename)
      inputs  = modelkrs.inputs
      outputs = modelkrs.layers[-2].output
      model   = tf.keras.Model(inputs=inputs, outputs=outputs)

    case "VIT":
      filename = _root_model + "_Representation_VIT" + "_k_{:.2f}".format(_k).replace('.','') + "_d_{:02d}".format(_d) + ".keras"
      

      modelkrs = tf.keras.models.load_model(filename, custom_objects={
          'PositionalEmbedding': PositionalEmbedding,
          'CLSToken': CLSToken
      }, safe_mode=False)
      
      latent_vector = modelkrs.get_layer("Latent_Space_Z").output
      
      model = tf.keras.Model(inputs=modelkrs.inputs, outputs=latent_vector)

    case "MAE_VIT":
      filename = _root_model + "_Representation_MAE_VIT" + "_k_{:.2f}".format(_k).replace('.','') + "_d_{:02d}".format(_d) + ".keras"
      

      modelkrs = tf.keras.models.load_model(filename, custom_objects={
          'PatchMasking': PatchMasking
      }, safe_mode=False)
      
      latent_vector = modelkrs.get_layer("Latent_Space_Z").output
      
      model = tf.keras.Model(inputs=modelkrs.inputs, outputs=latent_vector)

    case "DINO_VIT":

      filename = _root_model + "_Representation_DINO_VIT" + "_k_{:.2f}".format(_k).replace('.','') + "_d_{:02d}".format(_d) + ".keras"
      
      modelkrs = tf.keras.models.load_model(filename, custom_objects={
          'PositionalEmbedding': PositionalEmbedding,
          'CLSToken': CLSToken
      }, safe_mode=False)
      
      latent_vector = modelkrs.get_layer("Latent_Space").output
      
      model = tf.keras.Model(inputs=modelkrs.inputs, outputs=latent_vector)
    case _:
      print("Unknown representation model")
  #END_MATCH_MODEL_TYPE

  return model
#END_FUNCTION_LOAD_REPRESENTATION_MODEL

In [ ]:
def computeRepresentation(_X, _model, _model_type="PCA"):
  Z = []

  match _model_type:
    case "PCA":
      Z = _model.transform(_X)
    case "AE" | "AE_Patch"| "VIT" |"MAE_VIT"| "DINO_VIT":
      Z = _model.predict(_X, verbose=0)
    case _:
      print("Unknown representation model")
  #END_MATCH_MODEL_TYPE

  return Z
#END_FUNCTION_COMPUTE_REPRESENTATION

In [ ]:
def defineClassifier(_classifier_type="RF"):
  classifier = []

  match _classifier_type:
    case "NB":
      classifier = GaussianNB()
    case "SVM":
      classifier = SVC(gamma='auto', random_state=42)
    case "RF":
      classifier = RandomForestClassifier(random_state=42,n_jobs=25)
    case "DT":
      classifier = DecisionTreeClassifier(random_state=42)
    case "XGB":
      classifier = XGBClassifier(random_state=42,device='cuda')
    case _:
      print("Unkbown classifier")

  return classifier
#END_FUNCTION_DEFINE_CLASSIFIER

In [ ]:
def computePerformanceMetrics(_y_true, _y_pred, _class_labels):
  n_classes = np.unique(_class_labels).shape[0]
  cmatrix = confusion_matrix(y_true=_y_true, y_pred=_y_pred, labels=_class_labels)

  diag= np.eye(n_classes) * cmatrix
  OA  = np.sum(diag) / np.sum(cmatrix)
  AA  = np.sum(diag, 0) / np.sum(cmatrix, 0)
  AA[np.isnan(AA)] = 1.0
  AA  = np.mean(AA)
  f1_macro = f1_score(y_true=_y_true, y_pred=_y_pred, labels=_class_labels, average='macro', zero_division=1.0)
  f1_per_class = f1_score(y_true=_y_true, y_pred=_y_pred, labels=_class_labels, average=None, zero_division=1.0)

  return [OA, AA,f1_macro] + f1_per_class.tolist()
#END_FUNCTION_COMPUTE_PERFORMANCE_METRICS

# EXPERIMENTAL CONFIGURATIONS

In [ ]:
# DATASET CONFIGURATIONS
DATASET_PATH  = "Data/" #"/content/drive/MyDrive/Universidad_Pacifico/Investigación/2024_UP_Impact_Number_Samples_Representation/01_Datasets/"
DATASET_ID    = "indian"
DATASETS      = {"pavia":"Pavia_University", "botswana":"Botswana", "indian":"Indian_Pines", "ksc":"KSC", "salinas":"Salinas"}

# PREPROCESSING CONFGURATIONS
trn_steps     = 0.25
n_trn_subsets = int(1 / trn_steps)
K             = [trn_steps * k for k in range(1,n_trn_subsets + 1)]

# AUTOENCODER CONFIGURATIONS
#lambda_       = [0.000]

# REPRESENTATION CONFIGURATIONS
D             = [2 ** d for d in range(1,7)] #Latent dimensions

LEARNERS      = ["PCA","AE","AE_Patch","VIT","MAE_VIT","DINO_VIT"] 

# RESULTS CONFIGURATIONS
path_results     = "Results/"
path_models      = path_results + "Representation_Models/" + DATASETS[DATASET_ID] + "/"
path_performances= path_results + "Performances/Representation/"

createFolder(path_results)
createFolder(path_models)
createFolder(path_performances)
#----------------
#Clasification configurations
CLASSIFIERS   = ["XGB", "SVM", "RF", "DT", "NB"]


path_representation= path_results + "Representation_Models/" + DATASETS[DATASET_ID] + "/"
path_classification= path_results + "Classification_Models/" + DATASETS[DATASET_ID] + "/"
path_performances= path_results + "Performances/Classification/"

createFolder(path_representation)
createFolder(path_classification)
createFolder(path_performances)

## STAGE I: DATASET PREPARATION FOR REPRESENTATION LEARNING (DPRL)

In [ ]:
def prepare_representation_dataset(DATASET_ID, DATASET_PATH, DATASETS, Prepro, K, p_size=3,stand='max'):
    # LOAD A DATASET
    data_filename = DATASET_PATH + DATASETS[DATASET_ID] + ".mat"
    
    mat_I = scipy.io.loadmat(data_filename)
    I     = []  #Hyperspectral input image
    
    match DATASET_ID:
      case "pavia":
        I = mat_I["paviaU"].copy()
      case "botswana":
        I = mat_I["Botswana"].copy()
      case "indian":
        I = mat_I["indian_pines_corrected"].copy()
      case "ksc":
        I = mat_I["KSC"].copy()
      case "salinas":
        I = mat_I["salinas_corrected"].copy()
      case "houston":
        I = mat_I["Houston"].copy()
      case _:
        print("Unrecognized dataset")
    
    # -----
    try:
        del mat_I
    except NameError: pass
    gc.collect()
    # -------------------------------
    
    # Obtain input image properties
    M   = I.shape[0]    #M: rows
    N   = I.shape[1]    #N: cols
    B   = I.shape[2]    #B: bands
    Np  = M * N         #N_j: total number of pixels
    
    # Pixel-wise indexing
    pxl_cols = np.tile(np.arange(N), (M,1))
    pxl_rows = np.tile(np.arange(M), (N,1)).T
    pxl_indxs= N * pxl_rows + pxl_cols #pixel-based indexes, se tiene un indice unico por pixel
    
    # Tile-wise indexing
    tile_size_x = 32
    tile_size_y = 32
    n_tiles_x   = N // tile_size_x + 1
    n_tiles_y   = M // tile_size_y + 1
    n_tiles     = n_tiles_x * n_tiles_y
    
    tls_cols    = np.tile(np.sort(np.tile(np.arange(n_tiles_x), (1, tile_size_x))), (M,1))
    tls_rows    = np.tile(np.sort(np.tile(np.arange(n_tiles_y), (1, tile_size_y))), (N,1)).T
    
    tls_cols    = tls_cols[:M,:N]
    tls_rows    = tls_rows[:M,:N]
    tls_indxs   = n_tiles_x * tls_rows + tls_cols; #tile-based indexes, se tiene un indice por mosaico
    
    Indxs       = np.dstack((pxl_cols, pxl_rows, pxl_indxs, tls_indxs)) # indexing image, matriz 3d con 4 canales, te da el idx x, y, idx pixel (nro unico por pixel) y idx tile (nro del mosaico)
    

    n_T   = int(0.80 * n_tiles) 
    n_t   = n_tiles - n_T       
    
    np.random.seed(13)  # For reproducibility
    idxs   = np.arange(n_tiles)
    T_idxs = np.random.choice(idxs, size=n_T, replace=False, p=None)
    t_idxs = np.setdiff1d(idxs, T_idxs) 
    
    Indxs_flat = np.reshape(Indxs, newshape=(Np, 4), order='C')
    j_T_idxs   = np.full(Np, False)
    for T_idx in T_idxs:
      j_T_idxs |= (Indxs_flat[:,-1] == T_idx)
    j_t_idxs = ~j_T_idxs
    I_flat = np.reshape(I, newshape=(Np, B), order='C')
    
    if stand == 'max':
      train_max = I_flat[j_T_idxs].max() 
      I = I / train_max
    elif stand == 'pca_stand':
      mean_val = np.mean(I_flat[j_T_idxs], axis=0).reshape((1, 1, B))
      std_val  = np.std(I_flat[j_T_idxs], axis=0).reshape((1, 1, B))
      I = (I - mean_val) / (std_val + 1e-8)
    else:
      print("Estandarizacion no reconocida")
      return
    
    mask_2d = j_T_idxs.reshape(M, N)
    match Prepro:
        case "pixel":
            J_train = np.reshape(I, newshape=(Np, B), order='C')
            J_test  = J_train  
        case "patch":
            use_mean_pooling = False
            I_for_train = I.copy(); I_for_train[~mask_2d] = 0.0
            I_for_test  = I.copy(); I_for_test[mask_2d]   = 0.0
            J_train_raw = extract_spatial_patches(I_for_train, patch_size=p_size)
            J_test_raw  = extract_spatial_patches(I_for_test, patch_size=p_size)
            del I_for_train, I_for_test
            J_train = J_train_raw
            J_test  = J_test_raw
            del J_train_raw, J_test_raw
        case "3d_patch":
            I_for_train = I.copy(); I_for_train[~mask_2d] = 0.0
            I_for_test  = I.copy(); I_for_test[mask_2d]   = 0.0
            J_train = extract_spatial_patches_vectorized(I_for_train, patch_size=p_size)
            J_test  = extract_spatial_patches_vectorized(I_for_test, patch_size=p_size)
            del I_for_train, I_for_test
    
    
    Indxs = np.reshape(Indxs, newshape=(Np, 4), order='C') 
    
    T         = {}
    T['T']    = J_train[j_T_idxs]
    T['idxs'] = Indxs[j_T_idxs]
    
    t         = {}
    t['t']    = J_test[j_t_idxs]
    t['idxs'] = Indxs[j_t_idxs]
    
    T_k       = {}
    
    for k in K:
      n_T_k   = int(k*n_T)
      idxs_k  = np.random.choice(T_idxs, size=n_T_k, replace=False, p=None)
    
      #Tile-based indices
      T_k_idxs= np.full(Np, False)
      for idx_k in idxs_k:
        T_k_idxs= T_k_idxs | (Indxs[:,-1] == idx_k)
    
      key     = "T_" + str(k)
      T_k[key]= J_train[T_k_idxs,:]
      key     = "T_idxs_" + str(k)
      T_k[key]= Indxs[T_k_idxs,:]

    try:
        del J_train, J_test, Indxs, I
    except NameError: pass
    gc.collect()

    
    return T_k, t, B, p_size


## STAGE II: REPRESENTATION LEARNING (RL)

In [ ]:
G_k_d         = []                          
report        = []
_lambda = 0.00

pbar_extractors = tqdm(LEARNERS, desc="Extractores")
for extractor in pbar_extractors:

  # Preprocesamiento dependiendo del modelo
  stand="max"
  if extractor in ["PCA","AE"]:
      Prepro = "pixel"
      if extractor == 'PCA':
        stand="pca_stand"
  elif extractor in ["AE_Patch"]:
      Prepro = "patch"
  else: # "VIT", "MAE_VIT", "DINO_VIT"
      Prepro = "3d_patch"

  T_k, t, B, p_size = prepare_representation_dataset(DATASET_ID, DATASET_PATH, DATASETS, Prepro, K, p_size=3,stand=stand)
  pbar_k = tqdm(K, desc=f"k (Ext: {extractor})", leave=False)
  for k in pbar_k:
    pbar_d = tqdm(D, desc=f"d (k: {k})", leave=False)

    X_k_full    = T_k["T_" + str(k)]
    Idxs_k_full = T_k["T_idxs_" + str(k)]
    unique_tiles_k = np.unique(Idxs_k_full[:, -1])
    train_tiles, val_tiles = train_test_split(
        unique_tiles_k, test_size=0.20, random_state=42
    )
    mask_internal_train = np.isin(Idxs_k_full[:, -1], train_tiles)
    mask_internal_val   = np.isin(Idxs_k_full[:, -1], val_tiles)
    X_T_internal = X_k_full[mask_internal_train]
    X_v_internal = X_k_full[mask_internal_val]

    
    for d in pbar_d:
      G_k_d = []
      elapsed_time = 0
      pbar_extractors.set_postfix({"dataset": DATASETS[DATASET_ID], "d": d, "k": k})
      
      #Train representation models
      match extractor:

        case "PCA":
          filename= path_models + DATASETS[DATASET_ID] + "_Representation_{}".format(extractor) + "_k_{:.2f}".format(k).replace('.','') + "_d_{:02d}".format(d) + ".joblib"
          _lambda = 0.0
          

          G_k_d, elapsed_time = trainRepresentationPCA(X=X_T_internal, 
                                                       d=d,
                                                       name='PCA',
                                                       save_filename=filename)
          #
        
        case "AE":
          _lambda = 0.00 #Coefficient factor for HOAE. If 0.00 it behaves as a normal autoencoder

          filename= path_models + DATASETS[DATASET_ID] + "_Representation_{}".format("HOAE") + "_k_{:.2f}".format(k).replace('.','') + "_d_{:02d}".format(d) + "_f_{:.3f}".format(_lambda).replace('.','') + ".keras"
          G_k_d, elapsed_time, hist = trainRepresentationAE_original(X_T=X_T_internal,#X_T=T_k["T_" + str(k)],
                                                            X_v=X_v_internal,#t['t'],
                                                            input_dim=B,
                                                            encoder_dims=d,
                                                            lambda_=_lambda,
                                                            name=extractor,
                                                            save_filename=filename)
          

        case "AE_Patch":
          _lambda = 0.00 #Coefficient factor for HOAE. If 0.00 it behaves as a normal autoencoder
          
          filename= path_models + DATASETS[DATASET_ID] + "_Representation_{}".format("HOAE_Patch") + "_k_{:.2f}".format(k).replace('.','') + "_d_{:02d}".format(d) + "_f_{:.3f}".format(_lambda).replace('.','') + ".keras"
          G_k_d, elapsed_time, hist = trainRepresentationAE_original(X_T=X_T_internal,
                                                            X_v=X_v_internal,
                                                            input_dim=9*B,
                                                            encoder_dims=d,
                                                            lambda_=_lambda,
                                                            name=extractor,
                                                            save_filename=filename)

        case "VIT":
            _embed_dim = 64     
            _num_heads = 4      
            _ff_dim    = 128    
            _dropout   = 0.1
            _lambda    = 0.00 
            
            filename = path_models + DATASETS[DATASET_ID] + "_Representation_{}".format(extractor) + "_k_{:.2f}".format(k).replace('.','') + "_d_{:02d}".format(d) + ".keras"
            
            G_k_d, elapsed_time, hist = trainViT(X_T=X_T_internal,#
                                                 X_v=X_v_internal,
                                                 num_tokens=p_size*p_size,
                                                 input_dim=B,
                                                 embed_dim=_embed_dim,
                                                 num_heads=_num_heads,
                                                 ff_dim=_ff_dim,
                                                 dropout_rate=_dropout,
                                                 latent_dim=d,
                                                 name=extractor,
                                                 save_filename=filename)
            

        case "MAE_VIT":

            _embed_dim  = 128   
            _mask_ratio = 0.50  
            _lambda     = 0.00 
            
            filename = path_models + DATASETS[DATASET_ID] + "_Representation_{}".format(extractor) + "_k_{:.2f}".format(k).replace('.','') + "_d_{:02d}".format(d) + ".keras"
            
            G_k_d, elapsed_time, hist = trainMAE_ViT(X_T=X_T_internal,
                                                     X_v=X_v_internal,
                                                     num_tokens=p_size*p_size,
                                                     input_dim=B,
                                                     embed_dim=_embed_dim,
                                                     latent_dim=d, 
                                                     mask_ratio=_mask_ratio,
                                                     name=extractor,
                                                     save_filename=filename)
            

        case "DINO_VIT":

            _embed_dim = 128    
            _lambda    = 0.00   

            filename = path_models + DATASETS[DATASET_ID] + "_Representation_{}".format(extractor) + "_k_{:.2f}".format(k).replace('.','') + "_d_{:02d}".format(d) + ".keras"
            

            G_k_d, elapsed_time, hist = trainDINO_ViT(X_T=X_T_internal,
                                                     X_v=X_v_internal,
                                                     num_tokens=p_size*p_size,
                                                     input_dim=B,
                                                     encoder_dims=d, 
                                                     embed_dim=_embed_dim,
                                                     name=extractor,
                                                     save_filename=filename)
            
      
        case _:
          print('Unknow representation learning technique')
      #END_MATCH

      # Save report to csv
      report  = report + [[DATASETS[DATASET_ID], extractor, k, d, _lambda, elapsed_time]]

      columns = ['dataset', 'ftr_extractor', 'k', 'd', 'lambda', "elapsed_time"]
      report_1= pd.DataFrame(report, columns=columns)

      csv_filename = path_performances + DATASETS[DATASET_ID] + "_Processing_Times.csv"
      report_1.to_csv(csv_filename, sep=',', columns=columns)

      try:
          del G_k_d 
          del hist     
      except NameError:
          pass
          
      tf.keras.backend.clear_session()
      gc.collect()
    #END_FOR_D
    try:
        del X_T_internal, X_v_internal, X_k_full, Idxs_k_full
        del mask_internal_train, mask_internal_val
    except NameError: pass
    gc.collect()
  #END_FOR_K
  try:
      del T_k, t
  except NameError: pass
  gc.collect()
  
#END_FTR_EXTRACTORS

# CLASIFICATION

## STAGE I: DATASET PREPARATION FOR REPRESENTATION LEARNING (DPRL)

In [ ]:
def prepare_classification_dataset(DATASET_ID, DATASET_PATH, DATASETS, Prepro, p_size=3,stand='max'):
    # LOAD A DATASET
    data_filename = DATASET_PATH + DATASETS[DATASET_ID] + ".mat"
    gt_filename   = DATASET_PATH + DATASETS[DATASET_ID] + "_gt.mat"
    
    mat_I = scipy.io.loadmat(data_filename)
    mat_gt= scipy.io.loadmat(gt_filename)
    
    I     = []  #Hyperspectral input image
    GT    = []
    
    match DATASET_ID:
      case "pavia":
        I = mat_I["paviaU"].copy()
        gt= mat_gt["paviaU_gt"].copy()
      case "botswana":
        I = mat_I["Botswana"].copy()
        gt= mat_gt["Botswana_gt"].copy()
      case "indian":
        I = mat_I["indian_pines_corrected"].copy()
        gt= mat_gt["indian_pines_gt"].copy()
      case "ksc":
        I = mat_I["KSC"].copy()
        gt= mat_gt["KSC_gt"].copy()
      case "salinas":
        I = mat_I["salinas_corrected"].copy()
        gt= mat_gt["salinas_gt"].copy()
      case _:
        print("Unrecognized dataset")
    
    try:
        del mat_I, mat_gt
    except NameError: pass
    gc.collect()
    
    gt  = 1.0*gt - 1.0
    
    # Obtain input image properties
    M   = I.shape[0]    #M: rows
    N   = I.shape[1]    #N: cols
    B   = I.shape[2]    #B: bands
    Np  = M * N         #N_j: total number of pixels
    
    # Pixel-wise indexing
    pxl_cols = np.tile(np.arange(N), (M,1))
    pxl_rows = np.tile(np.arange(M), (N,1)).T
    pxl_indxs= N * pxl_rows + pxl_cols #pixel-based indexes
    
    # Tile-wise indexing
    tile_size_x = 32
    tile_size_y = 32
    n_tiles_x   = N // tile_size_x + 1
    n_tiles_y   = M // tile_size_y + 1
    n_tiles     = n_tiles_x * n_tiles_y
    
    tls_cols    = np.tile(np.sort(np.tile(np.arange(n_tiles_x), (1, tile_size_x))), (M,1))
    tls_rows    = np.tile(np.sort(np.tile(np.arange(n_tiles_y), (1, tile_size_y))), (N,1)).T
    
    tls_cols    = tls_cols[:M,:N]
    tls_rows    = tls_rows[:M,:N]
    tls_indxs   = n_tiles_x * tls_rows + tls_cols; #tile-based indexes
    
    Indxs       = np.dstack((pxl_cols, pxl_rows, pxl_indxs, tls_indxs)) # indexing image
    n_T   = int(0.80 * n_tiles) 
    n_t   = n_tiles - n_T       
    
    np.random.seed(13) 
    idxs   = np.arange(n_tiles)
    T_idxs = np.random.choice(idxs, size=n_T, replace=False, p=None)
    t_idxs = np.setdiff1d(idxs, T_idxs) 
    
    Indxs_flat = np.reshape(Indxs, newshape=(Np, 4), order='C')
    mask_T   = np.full(Np, False)
    for T_idx in T_idxs:
      mask_T |= (Indxs_flat[:,-1] == T_idx)
    mask_t = ~mask_T
    I_flat = np.reshape(I, newshape=(Np, B), order='C')
    
    if stand == 'max':
      train_max = I_flat[mask_T].max() 
      I = I / train_max
    elif stand == 'pca_stand':
      mean_val = np.mean(I_flat[mask_T], axis=0).reshape((1, 1, B))
      std_val  = np.std(I_flat[mask_T], axis=0).reshape((1, 1, B))
      I = (I - mean_val) / (std_val + 1e-8)
    else:
      print("Estandarizacion no reconocida")
      return

    mask_2d = mask_T.reshape(M, N)
    match Prepro:
        case "pixel":
            L_train = np.reshape(I, newshape=(Np, B), order='C')
            L_test  = L_train  
        case "patch":
            I_for_train = I.copy(); I_for_train[~mask_2d] = 0.0
            I_for_test  = I.copy(); I_for_test[mask_2d]   = 0.0
            L_train = extract_spatial_patches(I_for_train, patch_size=p_size)
            L_test  = extract_spatial_patches(I_for_test, patch_size=p_size)
            del I_for_train, I_for_test
        case "3d_patch":
            I_for_train = I.copy(); I_for_train[~mask_2d] = 0.0
            I_for_test  = I.copy(); I_for_test[mask_2d]   = 0.0
            L_train = extract_spatial_patches_vectorized(I_for_train, patch_size=p_size)
            L_test  = extract_spatial_patches_vectorized(I_for_test, patch_size=p_size)
            del I_for_train, I_for_test
    
    GT    = np.reshape(gt, newshape=(Np,1), order='C')
    Indxs = np.reshape(Indxs, newshape=(Np,4), order='C')
    
    
    T         = {}
    T['T']    = L_train[mask_T]
    T['y']    = GT[mask_T]
    T['idxs'] = Indxs[mask_T]
    
    t         = {}
    t['t']    = L_test[mask_t]
    t['y']    = GT[mask_t]
    t['idxs'] = Indxs[mask_t]
    
    if np.isin(T['idxs'][:,-1], t['idxs'][:,-1]).any():
      print("Warning: Training and testing sets overlap")
    
    n_folds    = 5
    n_tls_fold = n_T//n_folds
    
    np.random.seed(42)         #For reproducibility 42
    T_idxs = np.random.choice(T_idxs, size=n_T, replace=False, p=None)
    
    idxs_cv  = {}
    for f in range(1, n_folds + 1):
      start     = n_tls_fold*(f - 1)
      end       = n_tls_fold*(f)
      if f < n_folds:
        idxs_f  = T_idxs[start:end]
      else:
        idxs_f  = T_idxs[start:]
    
      idxs_cv[f] = idxs_f.copy()
    
    idxs_cv[0]= T_idxs.copy()  #full index set for easy to use cv

    try:
        del L_train, L_test, I, GT, Indxs
    except NameError: pass
    gc.collect()

    return T, t, idxs_cv, B, p_size, n_folds


## STAGE II: CLASSIFICATION LEARNING (CL)

In [ ]:
exe_type   = 3                            #1: train only | 2: evaluate only | 3: train and evaluate

pbar_classifiers = tqdm(CLASSIFIERS, desc="Clasificadores")
for classifier in pbar_classifiers:
  report  = []
  pbar_learners = tqdm(LEARNERS, desc=f"Clasificador ({classifier})", leave=False)
  for extractor in pbar_learners:
    # Definimos el tipo de preprocesamiento dependiendo de la técnica base
    stand="max"
    if extractor in ["PCA","AE"]:
        Prepro = "pixel"
        if extractor == "PCA":
          stand="pca_stand"
    elif extractor in ["AE_Patch"]:
        Prepro = "patch"
    else: 
        Prepro = "3d_patch"
    # Invocamos la función    T, t, idxs_cv, B, p_size, n_folds = prepare_classification_dataset(DATASET_ID, DATASET_PATH, DATASETS, Prepro, p_size=3,stand=stand)
    

    all_y = np.concatenate([T['y'], t['y']])
    global_classes = np.unique(all_y[all_y != -1.0])

    pbar_k = tqdm(K, desc="k", leave=False)
    for k in pbar_k:
      pbar_d = tqdm(D, desc="d", leave=False)      
      for d in pbar_d:
        pbar_f = tqdm(range(1, n_folds + 1), desc="Folds", leave=False)
        # Load representation model
        G_k_d   = loadRepresentationModel(_root_model=path_representation + DATASETS[DATASET_ID], _model_type=extractor, _k=k, _d=d, _lambda=0.00)
        
        # STAGE IV: FEATURE EXTRACTION
        Z_T = computeRepresentation(_X=T['T'], _model=G_k_d, _model_type=extractor)
        Z_t = computeRepresentation(_X=t['t'], _model=G_k_d, _model_type=extractor)

        # Cross-validation on projected data
        for f in pbar_f:
          pbar_f.set_postfix({"k": k, "d": d, "fold": f})

          # Cross-validation sets generation
          idxs_cv_v = idxs_cv[f]
          idxs_cv_T = np.setdiff1d(idxs_cv[0], idxs_cv_v)

          Np_T = T['idxs'].shape[0]
          mask_cv_v = np.full(Np_T, False)
          mask_cv_T = np.full(Np_T, False)

          for idx_cv_v in idxs_cv_v:
            mask_cv_v |=  (T['idxs'][:,-1] == idx_cv_v)
          mask_cv_T = ~mask_cv_v

          # Training and validation sets
          cv_Z_T    = Z_T[mask_cv_T]
          cv_y_T    = T['y'][mask_cv_T]

          cv_Z_v    = Z_T[mask_cv_v]
          cv_y_v    = T['y'][mask_cv_v]

          # Filter zero-valued class labels
          mask_y_T_0= cv_y_T[:,0] == -1.0
          mask_y_v_0= cv_y_v[:,0] == -1.0
          mask_y_t_0= t['y'][:,0] == -1.0

          cv_Z_T    = cv_Z_T[~mask_y_T_0]
          cv_y_T    = cv_y_T[~mask_y_T_0]

          cv_Z_v    = cv_Z_v[~mask_y_v_0]
          cv_y_v    = cv_y_v[~mask_y_v_0]

          cv_Z_t    = Z_t[~mask_y_t_0]
          cv_y_t    = t['y'][~mask_y_t_0]

          # Encode labels to prevent XGBoost ValueError
          from sklearn.preprocessing import LabelEncoder
          le = LabelEncoder()
          cv_y_T_enc = le.fit_transform(cv_y_T.ravel())

          # Train and sabe classifiers
          filename    = "{}{}_Representation_{}_k_{:.2f}_d_{:02d}_Classification_{}_fold_{:02d}".format(path_classification, DATASETS[DATASET_ID], extractor, k, d, classifier, f)
          filename    = filename.replace(".", "") + ".joblib"

          # Train classifier
          if exe_type == 1 or exe_type == 3:
            C_k_d = defineClassifier(_classifier_type=classifier)
            C_k_d.fit(X=cv_Z_T, y=cv_y_T_enc)

            # Save the classifier
            dump(C_k_d, filename)

            if exe_type == 1:
              #print("ok")
              continue
          else:
            # Load the classifier
            C_k_d = load(filename)
          #END_IF

          # Evaluate classifier performance
          classes = global_classes

          cv_T_scores = computePerformanceMetrics(cv_y_T, le.inverse_transform(C_k_d.predict(cv_Z_T)).reshape(-1, 1), classes);
          cv_v_scores = computePerformanceMetrics(cv_y_v, le.inverse_transform(C_k_d.predict(cv_Z_v)).reshape(-1, 1), classes);
          cv_t_scores = computePerformanceMetrics(cv_y_t, le.inverse_transform(C_k_d.predict(cv_Z_t)).reshape(-1, 1), classes);

          report      = report + [[DATASETS[DATASET_ID], k, d, extractor, classifier, f] + cv_T_scores + cv_v_scores + cv_t_scores]
          try:
              del C_k_d, cv_Z_T, cv_Z_v, cv_Z_t
          except NameError:
              pass
          gc.collect()
        #END_FOR_F

        
        base_columns = ['dataset', 'k', 'd', 'ftr_extractor', 'classifier', 'fold']
        f1_class_cols = [f'f1_c{i}' for i in range(len(classes))]
        metrics_suffixes = ['oa', 'aa', 'f1_macro'] + f1_class_cols
        
        trn_cols = [f'trn_{m}' for m in metrics_suffixes]
        val_cols = [f'val_{m}' for m in metrics_suffixes]
        tst_cols = [f'tst_{m}' for m in metrics_suffixes]
        
        columns = base_columns + trn_cols + val_cols + tst_cols

        report_1= pd.DataFrame(report, columns=columns)

        csv_filename = path_performances + DATASETS[DATASET_ID] + "_" + classifier + "_Performances.csv"
        report_1.to_csv(csv_filename, sep=',', columns=columns)
        try:
              del Z_T, Z_t, G_k_d
        except NameError:
              pass
        tf.keras.backend.clear_session()
        gc.collect()
      #END_FOR_D
    #END_FOR_K
    try:
        del T, t, idxs_cv
    except NameError: pass
    gc.collect()
  #END_FOR_LEARNERS
#END_FOR_CLASSIFIERS